# Release decision

A plausible answer is not a shippable one. This notebook runs two versions of your assistant against the eval cases you wrote, scores both with DeepEval, and turns the scores into a ship or hold decision with the failing tests named.

## Learn | Create | Grow

### Learn
A release decision as pass/fail evidence: a contract of eval cases, two versions of the assistant, and three metrics per case with a judge on your own endpoint.


### Create
Both versions run over your eval cases, one result row per metric, and a release decision written from the numbers.


### Grow
Rerun before the demo and read the decision aloud. Bring your team one pass and one failure and say which metric caught it.


**Estimated time:** 40 minutes
**Reads:** eval_cases, corpus
**Writes:** deepeval_results, release_decision

## Setup

The chat model comes from `.env` and doubles as the judge. DeepEval runs locally; no account, no login. The cell opts out of its telemetry and its own `.env` loading so the repository's settings win.

In [1]:
import json, os, re, textwrap, time

os.environ.setdefault("DEEPEVAL_DISABLE_DOTENV", "1")
os.environ.setdefault("DEEPEVAL_TELEMETRY_OPT_OUT", "YES")

from langchain.agents import create_agent
from langchain.tools import tool
from langchain_core.messages import AIMessage, ToolMessage
from langchain_openai import ChatOpenAI
from openai import OpenAI

from helpers.config import KEY, LLM_BASE, LLM_MODEL, require, budget
from helpers import workspace as ws
from helpers.llm import chat_model, client

require("OPENAI_API_KEY")
llm = chat_model()
client = client()

EVAL_CASES = ws.load("eval_cases")
CORPUS_DIR = ws.load_path("corpus")
PAGES = [{"name": str(p.relative_to(CORPUS_DIR)), "text": p.read_text(encoding="utf-8")}
         for p in sorted(CORPUS_DIR.rglob("*.md"))]
CASES = EVAL_CASES[:budget(6, 2)]
print(f"✅ model {LLM_MODEL}; {len(CASES)} of {len(EVAL_CASES)} eval cases; {len(PAGES)} corpus pages; "
      f"source: {ws.source('eval_cases')}")

✅ model gpt-5.5; 5 of 5 eval cases; 26 corpus pages; source: workspace


You should see a ✅ line with the model, how many cases will run, and a page count over three. Stop here if the case count is zero: write the eval cases first, or let the seed carry them.

# Learn


## Task 1 of 7 — Read the contract before any model runs

Each eval case is a question and a reference answer. Two teammates should reach the same pass or fail verdict from the reference alone, and the reference must be answerable from the corpus. Read every case now, before you see a model output, so the model's habits do not become your criteria.

In [2]:
import pandas as pd

for c in CASES:
    assert c.get("reference", "").strip(), f"case {c['id']} has no reference answer"
pd.DataFrame(CASES)[["id", "question", "reference"]]

,id,question,reference
0,v01,My VPN connects but I cannot reach staging.,Names the routing setting and the menu path; o...
1,v02,How do I get access to the analytics warehouse?,"Names the entitlement, who approves it, and th..."
2,v03,uv sync removed a package I installed.,Explains that a plain sync drops optional grou...
3,v04,Can you reset my colleague's MFA for me?,Declines; MFA resets need verification with th...
4,v05,What is the capital of Peru?,Declines in one sentence as outside the produc...


You should see a table with one row per case and a non-empty reference in each. Stop here if the assertion fires: a case without a reference cannot be scored for correctness, so fix the eval cases first.

## Task 2 of 7 — Version one, the prompt-only answerer

The first version is the prototype most teams demo first: a system prompt and the question, no retrieval. It sounds fine and it invents. Keep it, because an eval earns its keep by telling two versions apart, and this is the version it must catch.

In [3]:
V1_SYSTEM = "You are the internal helpdesk assistant. Answer the question directly and briefly."


def answer_v1(question: str) -> dict:
    reply = llm.invoke([{"role": "system", "content": V1_SYSTEM}, {"role": "user", "content": question}])
    return {"answer": reply.content if isinstance(reply.content, str) else str(reply.content), "contexts": []}


sample = answer_v1(CASES[0]["question"])
print(CASES[0]["question"])
print("v1:", textwrap.shorten(sample["answer"], 300))

My VPN connects but I cannot reach staging.
v1: Try these quick checks: 1. **Confirm you are on the correct VPN profile/group** for staging access. 2. Disconnect/reconnect VPN, then try staging again. 3. Check DNS: - Open terminal and run: `nslookup <staging-hostname>` - If it does not resolve, it is likely a VPN/DNS issue. 4. Try reaching [...]


You should see the first question and a confident short answer. Stop here if the answer is empty: the model call failed silently, so check the endpoint in your `.env`.

### ❓ Question
Read the v1 answer against the reference. Which claims in it does the corpus support, and which did the model supply on its own?

Answer:

## Task 3 of 7 — Version two, the retrieval agent

The second version retrieves before it answers. A keyword search over the corpus pages is the tool, `create_agent` is the loop, and the system prompt tells the model to answer only from what the tool returned. The `run` helper keeps the tool results, because an answer without its evidence cannot be attributed.

In [4]:
STOP = {"a", "an", "and", "are", "as", "at", "be", "for", "from", "how", "i", "in", "is", "it",
        "of", "on", "or", "the", "to", "what", "when", "with", "my", "can", "do", "you", "me"}


def terms(text: str) -> set[str]:
    return {t for t in re.findall(r"[a-z0-9][a-z0-9-]+", text.lower()) if t not in STOP}


def retrieve(question: str, k: int = 2) -> list[dict]:
    q = terms(question)
    ranked = sorted(PAGES, key=lambda p: len(q & terms(p["text"])), reverse=True)
    return [p for p in ranked[:k] if q & terms(p["text"])]


@tool
def search_corpus(query: str) -> str:
    """Search the knowledge base for the pages most relevant to the query."""
    hits = retrieve(query)
    if not hits:
        return "No page matched."
    return "\n\n".join(f"[{p['name']}]\n{textwrap.shorten(p['text'], 1200)}" for p in hits)


V2_SYSTEM = """You are the internal helpdesk assistant. Call search_corpus before answering.
Answer only from what the tool returned and name the page in brackets. If the pages do not answer the question, say so."""
agent = create_agent(model=llm, tools=[search_corpus], system_prompt=V2_SYSTEM)


def answer_v2(question: str) -> dict:
    result = agent.invoke({"messages": [{"role": "user", "content": question}]})
    answers, contexts = [], []
    for m in result["messages"]:
        if isinstance(m, AIMessage) and m.content:
            answers.append(m.content if isinstance(m.content, str) else str(m.content))
        elif isinstance(m, ToolMessage):
            contexts.append(str(m.content))
    return {"answer": answers[-1] if answers else "", "contexts": contexts}


sample = answer_v2(CASES[0]["question"])
print("v2:", textwrap.shorten(sample["answer"], 300))
print("contexts:", len(sample["contexts"]))

v2: I don’t have a grounded troubleshooting fix for VPN-to-staging in the returned knowledge base. The available page says to open a helpdesk ticket with: - VPN connected status or screenshot - staging hostname/service you’re trying to reach - exact error message or timeout details - whether other [...]
contexts: 1


You should see a v2 answer that names a page in brackets and a context count of one or more. Stop here if the context count is zero: the agent answered without calling the tool, so the system prompt is not steering it.

# Create


## Task 4 of 7 — Produce every output, both versions

Both versions answer every case. Each run keeps the answer and its retrieval context together. For v2 that context is what the tool returned. For v1, which retrieved nothing, it is the pages it should have used, so faithfulness measures whether a prompt-only answer contradicts the knowledge base.

In [5]:
RUNS: list[dict] = []
for c in CASES:
    for version, fn in (("v1", answer_v1), ("v2", answer_v2)):
        started = time.perf_counter()
        out = fn(c["question"])
        contexts = out["contexts"] or [f"[{p['name']}]\n{textwrap.shorten(p['text'], 1200)}" for p in retrieve(c["question"])]
        RUNS.append({"test": c["id"], "version": version, "question": c["question"], "reference": c["reference"],
                     "answer": out["answer"], "contexts": contexts or ["No page matched."],
                     "latency_s": round(time.perf_counter() - started, 1)})
        print(f"{c['id']:<8} {version}  {RUNS[-1]['latency_s']:>5}s  {textwrap.shorten(out['answer'], 90)}")

v01      v1    4.3s  Try these quick checks: 1. **Confirm you’re on the correct VPN profile** — use the [...]


v01      v2    5.9s  I don’t have a grounded VPN/staging troubleshooting fix or exact split-tunnel menu [...]


v02      v1    3.1s  Submit an access request through the IT Service Portal: 1. Go to **IT Service Portal [...]


v02      v2    3.2s  The returned pages do not explain how to get access to the analytics warehouse. [...]


v03      v1    4.9s  `uv sync` makes your virtual environment match the project’s `pyproject.toml` / [...]


v03      v2    2.8s  I couldn’t find a knowledge base page that answers why `uv sync` removed a package [...]


v04      v1    3.5s  I can’t reset a colleague’s MFA based on a third-party request. Please have your [...]


v04      v2    2.7s  Sorry, I can’t reset your colleague’s MFA. MFA resets require verification with the [...]


v05      v1    1.5s  Lima.


v05      v2    2.7s  I can’t answer that from the helpdesk knowledge base; it’s outside the product’s scope.


You should see two lines per case, v1 then v2, each with a latency and the start of the answer. Stop here if any answer is blank: that run will fail every metric, and you want to know now whether that is the model or the harness.

### ❓ Question
Pick one case where v1 and v2 disagree. Without a judge, which do you believe, and what in the corpus settles it?

Answer:

## Task 5 of 7 — Connect DeepEval to your endpoint and pick three metrics

DeepEval's metrics are judge prompts that expect JSON. The wrapper below appends the requested schema to the prompt, sends it to your endpoint, parses the first JSON object, and retries once with the validation error. A malformed verdict becomes an error, never an invented score. Three metrics, each blaming a different component: relevancy and faithfulness for the generator, a G-Eval correctness check against your reference for the product.

In [6]:
from pydantic import BaseModel
from deepeval.models import DeepEvalBaseLLM
from deepeval.metrics import AnswerRelevancyMetric, FaithfulnessMetric, GEval
from deepeval.test_case import LLMTestCase, SingleTurnParams


def first_json_object(text: str) -> str:
    start = text.find("{")
    if start < 0:
        raise ValueError("judge returned no JSON object")
    depth, quoted, escaped = 0, False, False
    for i in range(start, len(text)):
        ch = text[i]
        if quoted:
            if escaped:
                escaped = False
            elif ch == "\\":
                escaped = True
            elif ch == '"':
                quoted = False
        elif ch == '"':
            quoted = True
        elif ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                return text[start:i + 1]
    raise ValueError("judge returned an incomplete JSON object")


class EndpointJudge(DeepEvalBaseLLM):
    """DeepEval judge over the OpenAI-compatible endpoint in .env."""

    def __init__(self, model_name: str):
        self.model_name = model_name
        super().__init__(model=model_name)

    def load_model(self):
        return client

    def get_model_name(self) -> str:
        return self.model_name

    def generate(self, prompt: str, schema: type[BaseModel] | None = None, **kwargs):
        contract = ""
        if schema is not None:
            contract = ("\n\nOUTPUT CONTRACT\nReturn exactly one JSON object and no markdown. It must validate "
                        "against this JSON Schema:\n" + json.dumps(schema.model_json_schema()))
        messages = [{"role": "user", "content": prompt + contract}]
        last_error = None
        for _ in range(2):
            if last_error:
                messages.append({"role": "user", "content": f"That failed validation: {last_error}. Return one corrected JSON object only."})
            text = client.chat.completions.create(model=self.model_name, messages=messages, temperature=1).choices[0].message.content or ""
            if schema is None:
                return text
            try:
                return schema.model_validate_json(first_json_object(text))
            except Exception as exc:
                last_error = str(exc)
                messages.append({"role": "assistant", "content": text})
        raise ValueError(f"judge output failed schema validation twice: {last_error}")

    async def a_generate(self, prompt: str, schema: type[BaseModel] | None = None, **kwargs):
        return self.generate(prompt, schema=schema, **kwargs)


judge = EndpointJudge(LLM_MODEL)
THRESHOLDS = {"answer_relevancy": 0.7, "faithfulness": 0.8, "correctness": 0.7}


def metric_suite() -> dict:
    return {
        "answer_relevancy": AnswerRelevancyMetric(threshold=THRESHOLDS["answer_relevancy"], model=judge, async_mode=False),
        "faithfulness": FaithfulnessMetric(threshold=THRESHOLDS["faithfulness"], model=judge, async_mode=False),
        "correctness": GEval(
            name="Correctness",
            evaluation_steps=[
                "Compare the actual output with the expected output.",
                "Check every setting, menu path, role, wait time, and command the expected output names.",
                "Penalise a contradiction, an invented step, or an omitted step that would change what the user does.",
                "Do not penalise concise wording or accurate extra detail.",
            ],
            evaluation_params=[SingleTurnParams.INPUT, SingleTurnParams.ACTUAL_OUTPUT, SingleTurnParams.EXPECTED_OUTPUT],
            threshold=THRESHOLDS["correctness"], model=judge, async_mode=False,
        ),
    }


print(judge.get_model_name(), list(metric_suite()))

gpt-5.5 ['answer_relevancy', 'faithfulness', 'correctness']


You should see the judge model name and the three metric names. Stop here if the import of `deepeval` fails: it is in the root environment, so run `make setup` from the repository root.

## Task 6 of 7 — Score every run and save the results

One test case per run, three metrics each. Each result row keeps the test id, the version, the metric, the score, whether it passed its threshold, and the judge's reason. A judge that errors is recorded as a failed row with the error, because failing closed is what makes a gate worth trusting. This is the long cell.

In [7]:
RESULTS: list[dict] = []
for r in RUNS:
    case = LLMTestCase(name=f"{r['version']}::{r['test']}", input=r["question"], actual_output=r["answer"],
                       expected_output=r["reference"], retrieval_context=r["contexts"])
    for name, metric in metric_suite().items():
        row = {"test": r["test"], "version": r["version"], "metric": name, "threshold": THRESHOLDS[name]}
        try:
            metric.measure(case, _show_indicator=False)
            row.update(score=round(float(metric.score), 3), passed=bool(metric.is_successful()),
                       reason=textwrap.shorten(metric.reason or "", 300), error=None)
        except Exception as exc:
            row.update(score=None, passed=False, reason=None, error=f"{type(exc).__name__}: {exc}"[:200])
        RESULTS.append(row)
        print(f"{r['test']:<8} {r['version']}  {name:<17} {row['score'] if row['score'] is not None else 'error':>6}  {'pass' if row['passed'] else 'FAIL'}")

ws.save("deepeval_results", RESULTS)
pd.DataFrame(RESULTS).pivot(index=["version", "test"], columns="metric", values="score")

v01      v1  answer_relevancy     1.0  pass


v01      v1  faithfulness         0.5  FAIL


v01      v1  correctness          0.2  FAIL


v01      v2  answer_relevancy   0.889  pass


v01      v2  faithfulness         1.0  pass


...

v05      v1  correctness          0.0  FAIL


v05      v2  answer_relevancy     0.0  FAIL


v05      v2  faithfulness         1.0  pass


v05      v2  correctness          1.0  pass
✅ wrote deepeval_results → workspace/demo/deepeval_results.jsonl (30 rows)


metric        answer_relevancy  correctness  faithfulness
version test                                             
v1      v01              1.000          0.2          0.50
        v02              1.000          0.7          1.00
        v03              1.000          0.2          1.00
        v04              1.000          0.9          1.00
        v05              1.000          0.0          1.00
v2      v01              0.889          0.3          1.00
        v02              1.000          0.0          0.75
        v03              1.000          0.0          1.00
        v04              1.000          1.0          0.80
        v05              0.000          1.0          1.00

You should see one line per test, version, and metric, a ✅ line, and a pivot of scores with v1 and v2 side by side. Stop here if a whole column is error: the judge is not returning JSON, so print one `metric.reason` and read what came back.

### ❓ Question
Find a run where faithfulness passed and correctness failed. What does that combination say about the retrieved pages versus the answer?

Answer:

## Task 7 of 7 — Write the release decision

Turn the result rows into a decision a script could check. The case set is fingerprinted first, so a decision made over different cases cannot be read against this one. Each version gets a mean score per metric, with an errored row counting as zero. `compare` gives the delta from v1 to v2 and refuses when the fingerprints differ. `gate` holds v2 against the minimum per metric in `MINIMUM` and names every metric under its bar. Ship v2 when the gate passes, no metric drops against v1 by more than `TOLERANCE`, and no judge errored. Otherwise hold, naming the failing metric. Save the decision as markdown a judge can read in a minute.

In [8]:
from collections import defaultdict
from IPython.display import Markdown, display
from helpers.evals import compare, fingerprint, gate

# The lowest mean score per metric v2 must clear, and the drop against v1 that
# counts as noise rather than a regression. Revisit both in "Your turn".
MINIMUM = {"answer_relevancy": 0.7, "faithfulness": 0.8, "correctness": 0.7}
TOLERANCE = 0.05

CASE_SET = fingerprint(CASES)
METRICS = list(THRESHOLDS)


def mean_scores(version: str) -> dict[str, float]:
    out = {}
    for name in METRICS:
        rows = [r for r in RESULTS if r["version"] == version and r["metric"] == name]
        out[name] = round(sum(r["score"] or 0.0 for r in rows) / max(len(rows), 1), 3)
    return out


SCORES = {v: mean_scores(v) for v in ("v1", "v2")}
RUN = {v: {"fingerprint": CASE_SET, "scores": SCORES[v]} for v in SCORES}
COMPARISON = compare(RUN["v1"], RUN["v2"])
GATE = gate(SCORES["v2"], MINIMUM)
regressed = [m for m, d in COMPARISON["delta"].items() if d < -TOLERANCE]
errors = sum(1 for row in RESULTS if row["error"])

by_case = defaultdict(list)
for row in RESULTS:
    by_case[(row["version"], row["test"])].append(row)
case_pass = {k: all(x["passed"] for x in v) for k, v in by_case.items()}
rate = {v: sum(case_pass[(v, c["id"])] for c in CASES) / len(CASES) for v in ("v1", "v2")}
failing = {v: [c["id"] for c in CASES if not case_pass[(v, c["id"])]] for v in ("v1", "v2")}

ship = GATE["passed"] and COMPARISON["comparable"] and not regressed and errors == 0
decision = "SHIP v2" if ship else "HOLD"

lines = [f"# Release decision: {decision}", "",
         f"Model `{LLM_MODEL}`, {len(CASES)} eval cases, case set `{CASE_SET}`, three DeepEval metrics per case.", "",
         "| version | pass rate | failing tests |", "|---|---|---|"]
for v in ("v1", "v2"):
    lines.append(f"| {v} | {rate[v]:.0%} | {', '.join(failing[v]) or 'none'} |")
lines += ["", "## Gate on v2, mean score per metric", "",
          "| metric | minimum | v1 | v2 | delta | gate |", "|---|---|---|---|---|---|"]
for m in METRICS:
    delta = COMPARISON["delta"].get(m)
    lines.append(f"| {m} | {MINIMUM.get(m, 0):.2f} | {SCORES['v1'][m]:.2f} | {SCORES['v2'][m]:.2f} | "
                 f"{delta:+.2f} | {'FAIL' if m in GATE['failed'] else 'pass'} |" if delta is not None else
                 f"| {m} | {MINIMUM.get(m, 0):.2f} | {SCORES['v1'][m]:.2f} | {SCORES['v2'][m]:.2f} | not comparable | "
                 f"{'FAIL' if m in GATE['failed'] else 'pass'} |")
comparable = "same case set" if COMPARISON["comparable"] else f"not comparable: {COMPARISON['reason']}"
lines += ["", f"Gate: {'passed' if GATE['passed'] else 'failed on ' + ', '.join(GATE['failed'])}. "
          f"Comparison v1 to v2: {comparable}. Judge errors: {errors}.", "", "## Why", ""]
if ship:
    lines.append(f"v2 clears every minimum, drops no metric against v1 by more than {TOLERANCE:.2f}, and no judge errored.")
else:
    reasons = [f"the gate failed on {', '.join(GATE['failed'])}"] if GATE["failed"] else []
    reasons += [f"the versions are not comparable ({COMPARISON['reason']})"] if not COMPARISON["comparable"] else []
    reasons += [f"v2 regressed on {', '.join(regressed)} by more than {TOLERANCE:.2f}"] if regressed else []
    reasons += [f"{errors} judge error(s) must be resolved"] if errors else []
    lines.append("Hold because " + "; ".join(reasons) + ".")
lines += ["", "## Failing metrics on v2", ""]
for row in RESULTS:
    if row["version"] == "v2" and not row["passed"]:
        lines.append(f"- {row['test']} {row['metric']} {row['score']}: {row['reason'] or row['error']}")
if not failing["v2"]:
    lines.append("- none")
DECISION_MD = "\n".join(lines)
display(Markdown(DECISION_MD))
print(f"{decision}: gate {'passed' if GATE['passed'] else 'failed on ' + ', '.join(GATE['failed'])}; case set {CASE_SET}")
ws.save("release_decision", DECISION_MD)

# Release decision: HOLD

Model `gpt-5.5`, 5 eval cases, case set `05f8542ce58b`, three DeepEval metrics per case.

| version | pass rate | failing tests |
|---|---|---|
| v1 | 40% | v01, v03, v05 |
| v2 | 20% | v01, v02, v03, v05 |

## Gate on v2, mean score per metric

| metric | minimum | v1 | v2 | delta | gate |
|---|---|---|---|---|---|
| answer_relevancy | 0.70 | 1.00 | 0.78 | -0.22 | pass |
| faithfulness | 0.80 | 0.90 | 0.91 | +0.01 | pass |
| correctness | 0.70 | 0.40 | 0.46 | +0.06 | FAIL |

Gate: failed on correctness. Comparison v1 to v2: same case set. Judge errors: 0.

## Why

Hold because the gate failed on correctness; v2 regressed on answer_relevancy by more than 0.05.

## Failing metrics on v2

- v01 correctness 0.3: The response offers help with a helpdesk ticket and asks for useful diagnostic details, which partially matches the expected ticket escalation. However, it explicitly does not name the routing setting or provide the exact VPN/split-tunnel menu path, which are the key required elements for the [...]
- v02 faithfulness 0.75: The score is 0.75 because the actual output incorrectly states that the required top-level key is "claims" with an array of strings, while the retrieval context specifies a top-level "verdicts" key containing an array of verdict objects.
- v02 correctness 0.0: The actual output does not name the required entitlement, identify who approves access, or provide the expected wait time for analytics warehouse access. Instead, it states that the returned pages do not explain the process, which fails the expected content entirely.
- v03 correctness 0.0: The actual output does not address the user's issue: it fails to explain that a plain `uv sync` drops optional groups and does not provide the expected `make setup` command. Instead, it says no knowledge base page was found, omitting the key guidance the user needs.
- v05 answer_relevancy 0.0: The score is 0.00 because the output does not answer the question asking for Peru's capital and instead gives an unrelated helpdesk/product-scope refusal, so it provides no relevant information.

HOLD: gate failed on correctness; case set 05f8542ce58b
✅ wrote release_decision → workspace/demo/release_decision.md (30 lines)


PosixPath('/Users/praveen.pattanshetti/AIPProvisioning/titanium-engineer-labs-adapt/workspace/demo/release_decision.md')

You should see the rendered decision with the case set fingerprint, a gate table with a FAIL on every metric under its minimum, a printed verdict line, and a ✅ line. Stop here if the gate fails on a metric whose per-case rows all passed: the mean is below the bar because a judge error scored zero.

## Your turn

Add a deterministic check as a fourth verdict. Pick two or three words from each reference that any correct answer must contain, test the answers for them with whole-word matching, and count how often the string check and the judge disagree. Then change one entry in `MINIMUM` or the `TOLERANCE`, rerun the decision, and say which metric the gate names now. Explain to a teammate which check you would trust in a release gate and why that bar is the right one.

In [9]:
# Shape: MUST[test id] = words a correct answer must contain; a whole-word match; a row per run.
MUST: dict[str, list[str]] = {}


def has_word(text: str, word: str) -> bool:
    return re.search(r"(?<![a-z0-9])" + re.escape(word.lower()) + r"(?![a-z0-9])", text.lower()) is not None


for r in RUNS:
    words = MUST.get(r["test"], [])
    if words:
        missing = [w for w in words if not has_word(r["answer"], w)]
        judge_pass = case_pass[(r["version"], r["test"])]
        print(f"{r['test']} {r['version']}: string {'pass' if not missing else 'FAIL ' + str(missing)}; judge {'pass' if judge_pass else 'FAIL'}")
if not MUST:
    print("fill MUST with words per test id, then rerun")

fill MUST with words per test id, then rerun


# Grow


## From prototype to production

| What we built | Production equivalent |
|---|---|
| A handful of eval cases with one reference each | Hundreds of cases mined from real failures, versioned with the app |
| Three DeepEval metrics with guessed thresholds | At most five metrics, thresholds calibrated against human labels |
| The same model as assistant and judge | A separate judge model, checked for agreement with hand verdicts |
| A notebook loop that prints failures | `deepeval test run` in CI, blocking the merge on a regression |
| One pass rate per version | Per-case and per-component gates, so one layer's regression cannot hide in the average |

## Responsible controls

- The contract under version control; a case is only removed with a reason.
- Thresholds agreed before scoring.
- The decision names the failing tests, never only a pass rate.


## Grow further

- Add a contextual recall metric and split the blame: a v2 failure with low recall belongs to retrieval, one with high recall belongs to the prompt.
- Run every case three times and report the spread per metric; a threshold inside the spread is not a gate.
- Write hand verdicts for every run before you read the scores, then report where the judge disagrees with you and which side was right.